# Part A: Spam Classification using Naive Bayes

In this section, we will load the Lab 2 dataset, preprocess the text data using vectorization, and train two different Naive Bayes classifiers: **Gaussian Naive Bayes** and **Multinomial Naive Bayes**. Finally, we will compare their performance to determine which model is better suited for classifying spam.

In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import CountVectorizer
import warnings
warnings.filterwarnings('ignore')

# 1. Load the dataset
file_path = '../../practical_labs/Lab2_dataset.csv' 
df = pd.read_csv(file_path)

# Inspect the columns to ensure we use the correct names
print("Columns in dataset:", df.columns.tolist())
display(df.head())

# 2. Separate the text feature and target variable
text_col = 'text'
target_col = 'label'

X_text = df[text_col]
y = df[target_col]

# Encode the target variable ('ham' -> 0, 'spam' -> 1)
y = y.map({'ham': 0, 'spam': 1})

Columns in dataset: ['Unnamed: 0', 'label', 'text', 'label_num']


,Unnamed: 0,label,text,label_num
0,605,ham,Subject: enron methanol ; meter # : 988291\nth...,0
1,2349,ham,"Subject: hpl nom for january 9 , 2001\n( see a...",0
2,3624,ham,"Subject: neon retreat\nho ho ho , we ' re arou...",0
3,4685,spam,"Subject: photoshop , windows , office . cheap ...",1
4,2030,ham,Subject: re : indian springs\nthis deal is to ...,0


## Train/Test Split & Text Vectorization

We will split the dataset into an 80% training set and a 20% testing set. 
**Crucial Step:** We must split the data *before* vectorizing to prevent "data leakage" (ensuring the model doesn't learn vocabulary from the test set). We then use `CountVectorizer` to convert the raw text into numerical word counts.

In [9]:
# Split data FIRST
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42
)

# Vectorize the text data (Convert words to numbers)
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print(f"Training set size: {X_train.shape[0]} rows, {X_train.shape[1]} unique words (features)")
print(f"Testing set size: {X_test.shape[0]} rows, {X_test.shape[1]} unique words (features)")

Training set size: 4136 rows, 45240 unique words (features)
Testing set size: 1035 rows, 45240 unique words (features)


## Model 1: Gaussian Naive Bayes

Gaussian Naive Bayes assumes that the continuous features follow a normal (Gaussian) distribution. We will train the model and evaluate its accuracy on the test set.

In [11]:
# GaussianNB requires dense data, so we convert the sparse matrix to a dense array
X_train_dense = X_train.toarray()
X_test_dense = X_test.toarray()

# Initialize and train the Gaussian Naive Bayes model
gnb = GaussianNB()
gnb.fit(X_train_dense, y_train)

# Predict on the test set
y_pred_gnb = gnb.predict(X_test_dense)

# Evaluate the model
acc_gnb = accuracy_score(y_test, y_pred_gnb)
print(f"Gaussian Naive Bayes Accuracy: {acc_gnb * 100:.2f}%\n")
print("Classification Report (Gaussian NB):")
print(classification_report(y_test, y_pred_gnb, zero_division=0))

Gaussian Naive Bayes Accuracy: 95.46%

Classification Report (Gaussian NB):
              precision    recall  f1-score   support

           0       0.95      0.99      0.97       742
           1       0.96      0.87      0.92       293

    accuracy                           0.95      1035
   macro avg       0.96      0.93      0.94      1035
weighted avg       0.95      0.95      0.95      1035



## Model 2: Multinomial Naive Bayes

Multinomial Naive Bayes is specifically designed for discrete counts, such as the word counts generated by our `CountVectorizer`. It is the industry standard for text classification tasks like spam detection.

In [12]:
# Initialize and train the Multinomial Naive Bayes model
mnb = MultinomialNB()
mnb.fit(X_train, y_train)

# Predict on the test set
y_pred_mnb = mnb.predict(X_test)

# Evaluate the model
acc_mnb = accuracy_score(y_test, y_pred_mnb)
print(f"Multinomial Naive Bayes Accuracy: {acc_mnb * 100:.2f}%\n")
print("Classification Report (Multinomial NB):")
print(classification_report(y_test, y_pred_mnb, zero_division=0))

Multinomial Naive Bayes Accuracy: 97.20%

Classification Report (Multinomial NB):
              precision    recall  f1-score   support

           0       0.98      0.98      0.98       742
           1       0.96      0.95      0.95       293

    accuracy                           0.97      1035
   macro avg       0.97      0.96      0.97      1035
weighted avg       0.97      0.97      0.97      1035



## Model Comparison & Predictions

Now we will compare the accuracy of both models. Because Multinomial Naive Bayes is mathematically designed for word-count data, we expect it to outperform Gaussian Naive Bayes, which assumes a continuous bell-curve distribution.

In [13]:
# Compare the two models
print("--- Model Comparison ---")
print(f"Gaussian NB Accuracy:    {acc_gnb * 100:.2f}%")
print(f"Multinomial NB Accuracy: {acc_mnb * 100:.2f}%\n")

if acc_gnb > acc_mnb:
    print("Conclusion: Gaussian Naive Bayes performs better on this dataset.")
    best_model = gnb
    best_model_name = "Gaussian NB"
else:
    print("Conclusion: Multinomial Naive Bayes performs better on this dataset.")
    best_model = mnb
    best_model_name = "Multinomial NB"

# Use the best model to perform predictions on the test data
print(f"\n--- Sample Predictions using {best_model_name} ---")
predictions_df = pd.DataFrame({
    'Actual_Label': y_test.values,
    'Predicted_Label': best_model.predict(X_test)
})

# Map numbers back to text for easier reading in the output
predictions_df['Actual_Label'] = predictions_df['Actual_Label'].map({0: 'ham', 1: 'spam'})
predictions_df['Predicted_Label'] = predictions_df['Predicted_Label'].map({0: 'ham', 1: 'spam'})

# Display the first 15 predictions
display(predictions_df.head(15))

--- Model Comparison ---
Gaussian NB Accuracy:    95.46%
Multinomial NB Accuracy: 97.20%

Conclusion: Multinomial Naive Bayes performs better on this dataset.

--- Sample Predictions using Multinomial NB ---


,Actual_Label,Predicted_Label
0,ham,ham
1,spam,spam
2,ham,ham
3,ham,ham
4,ham,ham
5,ham,ham
6,ham,ham
7,ham,ham
8,ham,ham
9,ham,ham


# Part B: Airbnb Outlier Removal

In this section, we will analyze the Airbnb New York City dataset, specifically focusing on the nightly price column. Real estate and rental data is typically heavily right-skewed, meaning a few extremely high-priced luxury listings can drastically skew statistical measures like the mean. We will demonstrate and compare outlier removal techniques using Standard Deviation and the Interquartile Range (IQR) to produce a clean dataset.

In [15]:
# Load the Airbnb dataset
airbnb_file = '../../practical_labs/AB_NYC_2019.csv'
df_airbnb = pd.read_csv(airbnb_file)

print("Original Dataset Shape:", df_airbnb.shape)
display(df_airbnb.head())

# Define the price column
price_col = 'price'

# Clean the price column if it contains currency symbols or commas (e.g., '$1,200.00')
if df_airbnb[price_col].dtype == 'object':
    df_airbnb[price_col] = df_airbnb[price_col].replace({'\$': '', ',': ''}, regex=True).astype(float)

print(f"\nPrice Column Statistics:")
display(df_airbnb[price_col].describe())

Original Dataset Shape: (48895, 16)


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0



Price Column Statistics:


count    48895.000000
mean       152.720687
std        240.154170
min          0.000000
25%         69.000000
50%        106.000000
75%        175.000000
max      10000.000000
Name: price, dtype: float64

## Statistical Analysis & Outlier Removal

Rental prices are rarely normally distributed; they are usually right-skewed (a long tail of very expensive properties). Because of this, using the **Mean and Standard Deviation** to remove outliers is often ineffective, as the mean itself is pulled toward the expensive tail. 

Instead, the **Median and Interquartile Range (IQR)** is a much more robust method for skewed data, as the median is resistant to extreme values.

In [16]:
# --- Method 1: Mean and Standard Deviation ---
mean_price = df_airbnb[price_col].mean()
std_price = df_airbnb[price_col].std()

# Define outliers as values outside 3 standard deviations from the mean
lower_bound_mean = mean_price - (3 * std_price)
upper_bound_mean = mean_price + (3 * std_price)

df_clean_mean = df_airbnb[(df_airbnb[price_col] >= lower_bound_mean) & (df_airbnb[price_col] <= upper_bound_mean)]
print(f"Method 1 (Mean/Std): Removed {len(df_airbnb) - len(df_clean_mean)} outliers. Remaining rows: {len(df_clean_mean)}")


# --- Method 2: Median and IQR (Interquartile Range) ---
Q1 = df_airbnb[price_col].quantile(0.25)
Q3 = df_airbnb[price_col].quantile(0.75)
IQR = Q3 - Q1

# Define outliers as values below Q1 - 1.5*IQR or above Q3 + 1.5*IQR
lower_bound_iqr = Q1 - 1.5 * IQR
upper_bound_iqr = Q3 + 1.5 * IQR

df_clean_iqr = df_airbnb[(df_airbnb[price_col] >= lower_bound_iqr) & (df_airbnb[price_col] <= upper_bound_iqr)]
print(f"Method 2 (IQR): Removed {len(df_airbnb) - len(df_clean_iqr)} outliers. Remaining rows: {len(df_clean_iqr)}")

# Display the bounds for comparison
print(f"\n--- Bounds Comparison ---")
print(f"Mean/Std Bounds: ${lower_bound_mean:.2f} to ${upper_bound_mean:.2f}")
print(f"IQR Bounds:      ${lower_bound_iqr:.2f} to ${upper_bound_iqr:.2f}")

Method 1 (Mean/Std): Removed 388 outliers. Remaining rows: 48507
Method 2 (IQR): Removed 2972 outliers. Remaining rows: 45923

--- Bounds Comparison ---
Mean/Std Bounds: $-567.74 to $873.18
IQR Bounds:      $-90.00 to $334.00


## Conclusion & Clean Dataset Output

As seen in the bounds comparison, the Mean/Standard Deviation method results in a massive upper bound because the extreme luxury listings pull the mean and standard deviation upward. This method fails to filter out the high-end outliers effectively.

The IQR method, however, uses the median and quartiles, which are not affected by extreme values. It successfully isolates the core distribution of typical Airbnb prices and removes the extreme outliers, resulting in a much cleaner and more representative dataset.

Below is the final clean dataset using the IQR method.

In [17]:
# Display the first 10 rows of the clean dataset
print("Cleaned Dataset (using IQR method):")
display(df_clean_iqr.head(10))

# Optional: Save the clean dataset to a CSV file if you need to submit it separately
# df_clean_iqr.to_csv('airbnb_clean.csv', index=False)

Cleaned Dataset (using IQR method):


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0
5,5099,Large Cozy 1 BR Apartment In Midtown East,7322,Chris,Manhattan,Murray Hill,40.74767,-73.97500,Entire home/apt,200,3,74,2019-06-22,0.59,1,129
6,5121,BlissArtsSpace!,7356,Garon,Brooklyn,Bedford-Stuyvesant,40.68688,-73.95596,Private room,60,45,49,2017-10-05,0.40,1,0
7,5178,Large Furnished Room Near B'way,8967,Shunichi,Manhattan,Hell's Kitchen,40.76489,-73.98493,Private room,79,2,430,2019-06-24,3.47,1,220
8,5203,Cozy Clean Guest Room - Family Apt,7490,MaryEllen,Manhattan,Upper West Side,40.80178,-73.96723,Private room,79,2,118,2017-07-21,0.99,1,0
9,5238,Cute & Cozy Lower East Side 1 bdrm,7549,Ben,Manhattan,Chinatown,40.71344,-73.99037,Entire home/apt,150,1,160,2019-06-09,1.33,4,188


## Reflections/Observations

## Part A - Spam Classification
### On splitting before vectorizing
Vectorizing is the process of turning non-numerical data into numerical data for the purpose of training the model.

First, it's important to split before vectorizing. The reason this order is necessary is to prevent data leakage, which could inflate the reported accuracy. If I had vectorized the entire dataset first, the vocabulary and word frequencies would carry from the test set into training. This is what I initially referred to as data leakage.

### On why Gaussian NB is a poor choice for word counts
Gaussian NB assumes each feature is continuous and follows a roughly normal distribution. This is not true for word counts. Word counts have many zero values since most words don't appear in most emails. Still, Gaussian NB hit ~95% accuracy, which is a reminder that Naive Bayes is forgiving. 

### On why Multinomial NB wins
Multinomial NB is perfectly suited to this use case because it expects count data. The improvement is ~1.7 points overall and mainly shows up in spam recall. 


## Part B - Airbnb Outlier Removal
### On why the mean/std method underperforms here
The mean ($152.72) sits well above the median ($106) here and the std ($240) is larger than the mean. These are signatures of right-skewed distribution. The 3σ is at ~$873 which is above the 75th percentile of $175. This means the outlier threshold is being dragged upward by the outliers themselves. 

The lower bound from the mean/std method is negative, at ~-$568. That is meaningless for a price. That means the method is wasting half of its range on impossible values. 

### On why IQR works better
The IQR bounds ($-90 to $334) are asymmetric in a way that reflects the actual shape of the distribution. 
As far as outliers are concerned, it removed ~3000 rows vs 388 for the 3σ method. In this case, the model did a good job of removing the genuinely luxury-priced listings. 
The choice of outlier method depends on what the cleaned data will be used for. If the goal is to describe typical pricing, IQR is appropriate. 

